In [1]:
import pandas as pd
import seaborn as srn
import statistics  as sts
pd.options.display.float_format = '{:,.0f}'.format


In [2]:
dataset = pd.read_csv("../Balanco_ativo_Bruto/dfp_bp_2014.csv", encoding='utf-8', sep=";")
dataset.shape
dataset = dataset.drop(columns=['GRUPO_DFP','MOEDA'])
dataset = dataset.rename(columns={
    'CNPJ_CIA':      'cnpj',
    'DT_REFER':      'data_referencia',
    'VERSAO':        'versao',
    'DENOM_CIA':     'empresa',
    'CD_CVM':        'codigo_cvm',
    'ESCALA_MOEDA':  'escala',
    'ORDEM_EXERC':   'exercicio',
    'DT_FIM_EXERC':  'data_fim_exercicio',
    'CD_CONTA':      'codigo_conta',
    'DS_CONTA':      'descricao_conta',
    'VL_CONTA': 'valor',
    'ST_CONTA_FIXA': 'conta_fixa',
})
dataset['exercicio'] = dataset['exercicio'].map({'ÚLTIMO': 'atual', 'PENÚLTIMO': 'anterior'})
dataset = dataset[dataset.exercicio == 'atual']

dataset = dataset.drop(columns=['exercicio', 'versao'])

dataset.head(10)

,cnpj,data_referencia,empresa,codigo_cvm,escala,data_fim_exercicio,codigo_conta,descricao_conta,valor,conta_fixa
1,00.000.000/0001-91,2014-12-31,BCO BRASIL S.A.,1023,MIL,2014-12-31,1,Ativo Total,"1,278,136,948",S
3,00.000.000/0001-91,2014-12-31,BCO BRASIL S.A.,1023,MIL,2014-12-31,1.01,Caixa e Equivalentes de Caixa,"62,078,581",S
5,00.000.000/0001-91,2014-12-31,BCO BRASIL S.A.,1023,MIL,2014-12-31,1.01.01,Caixa e Depósitos Bancários,"13,337,180",N
7,00.000.000/0001-91,2014-12-31,BCO BRASIL S.A.,1023,MIL,2014-12-31,1.01.02,Depósitos Interfinanceiros,"35,592,524",N
9,00.000.000/0001-91,2014-12-31,BCO BRASIL S.A.,1023,MIL,2014-12-31,1.01.03,Aplicações em Operações Compromissadas,"13,148,877",N
11,00.000.000/0001-91,2014-12-31,BCO BRASIL S.A.,1023,MIL,2014-12-31,1.01.04,Aplicações em Moeda Estrangeira,0,N
13,00.000.000/0001-91,2014-12-31,BCO BRASIL S.A.,1023,MIL,2014-12-31,1.02,Aplicações Financeiras,"106,605,119",S
15,00.000.000/0001-91,2014-12-31,BCO BRASIL S.A.,1023,MIL,2014-12-31,1.02.01,Aplicações Financeiras Avaliadas a Valor Justo,"106,245,244",S
17,00.000.000/0001-91,2014-12-31,BCO BRASIL S.A.,1023,MIL,2014-12-31,1.02.01.01,Títulos para Negociação,"12,441,262",S
19,00.000.000/0001-91,2014-12-31,BCO BRASIL S.A.,1023,MIL,2014-12-31,1.02.01.02,Títulos Disponíveis para Venda,"93,803,982",S


In [3]:
empresas_selecionadas = {
    # FINANCEIRO
    'BCO BRASIL S.A.':              ('Financeiro', 'Estatal'),
    'ITAU UNIBANCO HOLDING S.A.':   ('Financeiro', 'Privado'),
    'BCO BRADESCO S.A.':            ('Financeiro', 'Privado'),
    'BCO SANTANDER (BRASIL) S.A.':  ('Financeiro', 'Privado'),
    'BCO BTG PACTUAL S.A.':         ('Financeiro', 'Privado'),
    'BCO ABC BRASIL S.A.':          ('Financeiro', 'Privado'),
    'BCO PAN S.A.':                 ('Financeiro', 'Privado'),

    # ENERGIA
    'CIA ENERGETICA DE MINAS GERAIS - CEMIG': ('Energia', 'Estatal'),
    'CIA PARANAENSE DE ENERGIA - COPEL':       ('Energia', 'Estatal'),
    'CENTRAIS ELET BRAS S.A. - ELETROBRAS':   ('Energia', 'Estatal'),
    'ENGIE BRASIL ENERGIA S.A.':              ('Energia', 'Privado'),
    'EQUATORIAL ENERGIA S.A.':                ('Energia', 'Privado'),
    'CPFL ENERGIA S.A.':                      ('Energia', 'Privado'),
    'ENERGISA S.A.':                          ('Energia', 'Privado'),
    'ALUPAR INVESTIMENTO S/A':                ('Energia', 'Privado'),
    'NEOENERGIA S.A.':                        ('Energia', 'Privado'),

    # PETRÓLEO
    'PETROLEO BRASILEIRO S.A. PETROBRAS': ('Petroleo', 'Estatal'),
    'PRIO S.A.':                           ('Petroleo', 'Privado'),
}

# Filtrar
dataset = dataset[dataset['empresa'].isin(empresas_selecionadas.keys())].copy()

# Adicionar colunas de setor e tipo
dataset['setor'] = dataset['empresa'].map(lambda x: empresas_selecionadas[x][0]) # type: ignore
dataset['tipo']  = dataset['empresa'].map(lambda x: empresas_selecionadas[x][1]) # type: ignore

# Conferir resultado
print(dataset.groupby(['setor', 'tipo', 'empresa']).size().reset_index(name='linhas'))

dataset.shape

         setor     tipo                                 empresa  linhas
0      Energia  Estatal    CENTRAIS ELET BRAS S.A. - ELETROBRAS      83
1      Energia  Estatal  CIA ENERGETICA DE MINAS GERAIS - CEMIG      73
2      Energia  Estatal       CIA PARANAENSE DE ENERGIA - COPEL      79
3      Energia  Privado                 ALUPAR INVESTIMENTO S/A      77
4      Energia  Privado                       CPFL ENERGIA S.A.      72
5      Energia  Privado                           ENERGISA S.A.      73
6      Energia  Privado               ENGIE BRASIL ENERGIA S.A.      70
7      Energia  Privado                 EQUATORIAL ENERGIA S.A.      79
8      Energia  Privado                         NEOENERGIA S.A.      81
9   Financeiro  Estatal                         BCO BRASIL S.A.      34
10  Financeiro  Privado                     BCO ABC BRASIL S.A.      30
11  Financeiro  Privado                       BCO BRADESCO S.A.      31
12  Financeiro  Privado                    BCO BTG PACTUAL S.A. 

(1044, 12)

In [4]:
dataset.to_csv("Balanco_tratado/dfp_bp_2014_filtrado.csv", index=False, encoding='utf-8', sep=";")

OSError: Cannot save file into a non-existent directory: 'Balanco_tratado'